In [1]:
!pip install yfinance pandas

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------------- -- 1.6/1.7 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 7.4 MB/s  0:00:00

   ---------------------------------------- 0/4 [peewee]
   ---------------------------------------- 0/4 [peewee]
   ---------------------------------------- 0/4 [peewee]
   ---------------------------------------- 0/4 [peewee]
   -------------------- ------------------- 2/4 [curl_cffi]
   -------------------- ------------------- 2/4 [curl_cffi]
   -------------------- ------------------- 2/4 [curl_cffi]
   ------------------------------ --------- 3/4 [yfinance]
   ------------------------------ --------- 3/4 [yfinance]
   ------------------------------ --------- 3/4 [yfinance]
   ------------------------------ --------- 3/4 [yfinance]
   ------------------------------ --------- 3/4 [yfinance]
   ---------------------------------------- 4/4 [yfinance]




[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd

In [3]:
TICKERS = {
    "GC=F": "gold_usd",       # Altın vadeli işlem (ons/USD)
    "DX-Y.NYB": "usd_index",  # Dolar endeksi (DXY)
    "^GSPC": "sp500",         # S&P500 - risk iştahı göstergesi
    "CL=F": "oil_usd",        # Ham petrol
    "USDTRY=X": "usd_try",    # Dolar/TL kuru
}

In [4]:
def veri_cek(baslangic="2015-01-01"):
    """Her sembolü tek tek çeker ve tek bir DataFrame'de birleştirir."""
    veriler = {}

    for sembol, isim in TICKERS.items():
        print(f"Çekiliyor: {sembol} ({isim})...")
        ticker = yf.Ticker(sembol)
        hist = ticker.history(start=baslangic, interval="1d")
        print(f"  -> {len(hist)} satır geldi")

        seri = hist["Close"]
        # Saat/saat dilimi farklarını atıp sadece tarihe indirgiyoruz
        seri.index = seri.index.tz_localize(None).normalize()
        # Aynı tarihte birden fazla kayıt varsa sonuncusunu al
        seri = seri[~seri.index.duplicated(keep="last")]

        veriler[isim] = seri

    # Hepsini tarih (index) üzerinden birleştir
    birlesik = pd.DataFrame(veriler)

    # Hafta sonları piyasalar kapalı olduğu için NaN satırlar oluşabilir,
    # bir önceki değeri ileri taşıyarak (forward fill) dolduruyoruz
    birlesik = birlesik.ffill()

    # Ons altın (USD) -> gram altın (TL) çevirisi
    birlesik["gram_altin_try"] = (birlesik["gold_usd"] / 31.1035) * birlesik["usd_try"]

    return birlesik

In [5]:
df = veri_cek()
df.tail(10)

Çekiliyor: GC=F (gold_usd)...
  -> 2905 satır geldi
Çekiliyor: DX-Y.NYB (usd_index)...
  -> 2906 satır geldi
Çekiliyor: ^GSPC (sp500)...
  -> 2906 satır geldi
Çekiliyor: CL=F (oil_usd)...
  -> 2906 satır geldi
Çekiliyor: USDTRY=X (usd_try)...
  -> 3010 satır geldi


,gold_usd,usd_index,sp500,oil_usd,usd_try,gram_altin_try
Date,,,,,,
2026-07-14,4061.100098,100.940002,7543.589844,79.339996,47.022598,6139.613819
2026-07-15,4044.000000,100.500000,7572.399902,79.599998,47.025799,6114.177836
2026-07-16,3985.600098,100.730003,7533.770020,78.949997,47.036598,6027.266076
2026-07-17,4012.699951,100.750000,7457.689941,82.489998,47.102100,6076.698631
2026-07-20,4010.300049,100.989998,7443.279785,83.230003,47.141140,6078.097834
2026-07-21,4071.100098,101.180000,7509.200195,84.910004,47.180599,6175.412480
2026-07-22,4146.899902,101.139999,7498.959961,86.830002,47.197701,6292.672516
2026-07-23,4046.600098,101.430000,7408.299805,92.190002,47.212399,6142.385793
2026-07-24,4067.600098,101.430000,7408.299805,89.309998,47.316799,6187.915087


In [6]:
df.tail()


,gold_usd,usd_index,sp500,oil_usd,usd_try,gram_altin_try
Date,,,,,,
2026-07-21,4071.100098,101.180000,7509.200195,84.910004,47.180599,6175.412480
2026-07-22,4146.899902,101.139999,7498.959961,86.830002,47.197701,6292.672516
2026-07-23,4046.600098,101.430000,7408.299805,92.190002,47.212399,6142.385793
2026-07-24,4067.600098,101.430000,7408.299805,89.309998,47.316799,6187.915087
2026-07-25,4067.600098,101.430000,7408.299805,89.309998,47.330002,6189.641682


In [7]:
print(df.isna().sum())
print(f"\nToplam satır: {len(df)}")
print(f"Tarih aralığı: {df.index.min()} - {df.index.max()}")

gold_usd          1
usd_index         1
sp500             1
oil_usd           1
usd_try           0
gram_altin_try    1
dtype: int64

Toplam satır: 3014
Tarih aralığı: 2015-01-01 00:00:00 - 2026-07-25 00:00:00


In [8]:
df = df.dropna()
print(f"Temizlik sonrası satır sayısı: {len(df)}")

df.to_csv("altin_verisi.csv")
print("Kaydedildi: altin_verisi.csv")

Temizlik sonrası satır sayısı: 3013
Kaydedildi: altin_verisi.csv


In [9]:
!pip install sqlalchemy


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

with engine.connect() as conn:
    print("Bağlantı başarılı!")

Bağlantı başarılı!


In [11]:
# DataFrame'i doğrudan PostgreSQL'e tablo olarak yazıyoruz
df.to_sql("altin_fiyatlari", engine, if_exists="replace", index=True, index_label="tarih")

print("Tablo oluşturuldu: altin_fiyatlari")

Tablo oluşturuldu: altin_fiyatlari


In [12]:
kontrol = pd.read_sql("SELECT * FROM altin_fiyatlari ORDER BY tarih DESC LIMIT 5", engine)
kontrol

,tarih,gold_usd,usd_index,sp500,oil_usd,usd_try,gram_altin_try
0,2026-07-25,4067.600098,101.430000,7408.299805,89.309998,47.330002,6189.641682
1,2026-07-24,4067.600098,101.430000,7408.299805,89.309998,47.316799,6187.915087
2,2026-07-23,4046.600098,101.430000,7408.299805,92.190002,47.212399,6142.385793
3,2026-07-22,4146.899902,101.139999,7498.959961,86.830002,47.197701,6292.672516
4,2026-07-21,4071.100098,101.180000,7509.200195,84.910004,47.180599,6175.412480


In [13]:
# Dünkü ve geçen haftaki gram altın fiyatı
df["gram_altin_lag1"] = df["gram_altin_try"].shift(1)   # 1 gün önce
df["gram_altin_lag7"] = df["gram_altin_try"].shift(7)   # 7 gün önce

df[["gram_altin_try", "gram_altin_lag1", "gram_altin_lag7"]].tail(10)

,gram_altin_try,gram_altin_lag1,gram_altin_lag7
Date,,,
2026-07-14,6139.613819,6038.073123,6178.074644
2026-07-15,6114.177836,6139.613819,6253.002250
2026-07-16,6027.266076,6114.177836,6239.706819
2026-07-17,6076.698631,6027.266076,6132.087636
2026-07-20,6078.097834,6076.698631,6224.166900
2026-07-21,6175.412480,6078.097834,6194.738927
2026-07-22,6292.672516,6175.412480,6038.073123
2026-07-23,6142.385793,6292.672516,6139.613819
2026-07-24,6187.915087,6142.385793,6114.177836


In [14]:
# 7 ve 30 günlük hareketli ortalama
df["ma_7"] = df["gram_altin_try"].rolling(window=7).mean()
df["ma_30"] = df["gram_altin_try"].rolling(window=30).mean()

df[["gram_altin_try", "ma_7", "ma_30"]].tail(10)

,gram_altin_try,ma_7,ma_30
Date,,,
2026-07-14,6139.613819,6174.484211,6232.959509
2026-07-15,6114.177836,6154.652151,6218.307033
2026-07-16,6027.266076,6124.303474,6198.694602
2026-07-17,6076.698631,6116.390759,6187.142508
2026-07-20,6078.097834,6095.523749,6175.572240
2026-07-21,6175.412480,6092.762828,6170.924123
2026-07-22,6292.672516,6129.134170,6177.626424
2026-07-23,6142.385793,6129.530167,6180.068444
2026-07-24,6187.915087,6140.064059,6177.454429


In [15]:
# Son 7 günün standart sapması - piyasa ne kadar "sakin" ya da "dalgalı"
df["volatilite_7"] = df["gram_altin_try"].rolling(window=7).std()

df[["gram_altin_try", "volatilite_7"]].tail(10)

,gram_altin_try,volatilite_7
Date,,
2026-07-14,6139.613819,76.287934
2026-07-15,6114.177836,70.282401
2026-07-16,6027.266076,73.238555
2026-07-17,6076.698631,75.222632
2026-07-20,6078.097834,58.812262
2026-07-21,6175.412480,53.601138
2026-07-22,6292.672516,86.555656
2026-07-23,6142.385793,86.617910
2026-07-24,6187.915087,88.893512


In [16]:
# Hedef 1: Yarının gram altın fiyatı (regresyon için)
df["target_fiyat"] = df["gram_altin_try"].shift(-1)

# Hedef 2: Yarın fiyat artacak mı? (sınıflandırma için) - 1 = artış, 0 = düşüş
df["target_yon"] = (df["target_fiyat"] > df["gram_altin_try"]).astype(int)

df[["gram_altin_try", "target_fiyat", "target_yon"]].tail(10)

,gram_altin_try,target_fiyat,target_yon
Date,,,
2026-07-14,6139.613819,6114.177836,0
2026-07-15,6114.177836,6027.266076,0
2026-07-16,6027.266076,6076.698631,1
2026-07-17,6076.698631,6078.097834,1
2026-07-20,6078.097834,6175.412480,1
2026-07-21,6175.412480,6292.672516,1
2026-07-22,6292.672516,6142.385793,0
2026-07-23,6142.385793,6187.915087,1
2026-07-24,6187.915087,6189.641682,1


In [17]:
# Modelin eğitileceği "tam" veri (NaN olmayan satırlar)
df_model = df.dropna()

# Bugünün satırını ayrı tutuyoruz - gerçek tahmin için kullanacağız
bugun = df.iloc[[-1]]

print(f"Eğitim için kullanılabilir satır: {len(df_model)}")
print(f"Bugünün satırı (tahmin için ayrıldı): {bugun.index[0]}")

df_model.tail()

Eğitim için kullanılabilir satır: 2983
Bugünün satırı (tahmin için ayrıldı): 2026-07-25 00:00:00


,gold_usd,usd_index,sp500,oil_usd,usd_try,gram_altin_try,gram_altin_lag1,gram_altin_lag7,ma_7,ma_30,volatilite_7,target_fiyat,target_yon
Date,,,,,,,,,,,,,
2026-07-20,4010.300049,100.989998,7443.279785,83.230003,47.141140,6078.097834,6076.698631,6224.166900,6095.523749,6175.572240,58.812262,6175.412480,1
2026-07-21,4071.100098,101.180000,7509.200195,84.910004,47.180599,6175.412480,6078.097834,6194.738927,6092.762828,6170.924123,53.601138,6292.672516,1
2026-07-22,4146.899902,101.139999,7498.959961,86.830002,47.197701,6292.672516,6175.412480,6038.073123,6129.134170,6177.626424,86.555656,6142.385793,0
2026-07-23,4046.600098,101.430000,7408.299805,92.190002,47.212399,6142.385793,6292.672516,6139.613819,6129.530167,6180.068444,86.617910,6187.915087,1
2026-07-24,4067.600098,101.430000,7408.299805,89.309998,47.316799,6187.915087,6142.385793,6114.177836,6140.064059,6177.454429,88.893512,6189.641682,1


In [18]:
# Güncellenmiş tabloyu (tüm feature'larla) PostgreSQL'e yazıyoruz
df.to_sql("altin_fiyatlari", engine, if_exists="replace", index=True, index_label="tarih")

print("Tablo güncellendi: altin_fiyatlari (feature'lar dahil)")

Tablo güncellendi: altin_fiyatlari (feature'lar dahil)


In [19]:
!pip install scikit-learn


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Kullanacağımız feature'lar (girdi)
feature_kolonlari = [
    "gold_usd", "usd_index", "sp500", "oil_usd", "usd_try",
    "gram_altin_lag1", "gram_altin_lag7",
    "ma_7", "ma_30", "volatilite_7"
]

X = df_model[feature_kolonlari]
y_fiyat = df_model["target_fiyat"]      # regresyon hedefi
y_yon = df_model["target_yon"]          # sınıflandırma hedefi

print(f"Feature sayısı: {X.shape[1]}")
print(f"Toplam örnek: {X.shape[0]}")
X.head()

Feature sayısı: 10
Toplam örnek: 2983


,gold_usd,usd_index,sp500,oil_usd,usd_try,gram_altin_lag1,gram_altin_lag7,ma_7,ma_30,volatilite_7
Date,,,,,,,,,,
2015-02-12,1220.099976,94.089996,2088.479980,51.209999,2.49811,97.756322,98.446417,98.052655,95.305229,0.977520
2015-02-13,1226.500000,94.199997,2096.989990,52.779999,2.46750,97.993603,97.581613,98.012508,95.585847,1.005462
2015-02-16,1226.500000,94.199997,2096.989990,52.779999,2.45597,97.300584,99.810125,97.589051,95.792067,0.699999
2015-02-17,1208.099976,94.059998,2100.340088,53.529999,2.45486,96.845926,96.630480,97.406114,95.924777,1.064626
2015-02-18,1199.699951,94.199997,2099.679932,52.139999,2.45150,95.349921,98.604979,96.827880,96.062063,1.362492


In [21]:
# Zaman serisi olduğu için rastgele DEĞİL, tarihe göre bölüyoruz
bolme_noktasi = int(len(X) * 0.8)

X_train, X_test = X.iloc[:bolme_noktasi], X.iloc[bolme_noktasi:]
y_fiyat_train, y_fiyat_test = y_fiyat.iloc[:bolme_noktasi], y_fiyat.iloc[bolme_noktasi:]
y_yon_train, y_yon_test = y_yon.iloc[:bolme_noktasi], y_yon.iloc[bolme_noktasi:]

print(f"Eğitim seti: {len(X_train)} satır ({X_train.index.min()} - {X_train.index.max()})")
print(f"Test seti:   {len(X_test)} satır ({X_test.index.min()} - {X_test.index.max()})")

Eğitim seti: 2386 satır (2015-02-12 00:00:00 - 2024-04-04 00:00:00)
Test seti:   597 satır (2024-04-05 00:00:00 - 2026-07-24 00:00:00)


In [22]:
# Regresyon modeli - yarının fiyatını tahmin eder
model_fiyat = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
model_fiyat.fit(X_train, y_fiyat_train)

# Sınıflandırma modeli - yarın artar mı azalır mı tahmin eder
model_yon = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
model_yon.fit(X_train, y_yon_train)

print("Her iki model de eğitildi!")

Her iki model de eğitildi!


In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score
import numpy as np

# --- Regresyon değerlendirmesi ---
tahmin_fiyat = model_fiyat.predict(X_test)

mae = mean_absolute_error(y_fiyat_test, tahmin_fiyat)
rmse = np.sqrt(mean_squared_error(y_fiyat_test, tahmin_fiyat))

print("=== REGRESYON (Fiyat Tahmini) ===")
print(f"MAE  (Ortalama Mutlak Hata): {mae:.2f} TL")
print(f"RMSE (Kök Ortalama Kare Hata): {rmse:.2f} TL")
print(f"Karşılaştırma - ortalama gram altın fiyatı: {y_fiyat_test.mean():.2f} TL")

# --- Sınıflandırma değerlendirmesi ---
tahmin_yon = model_yon.predict(X_test)

acc = accuracy_score(y_yon_test, tahmin_yon)
f1 = f1_score(y_yon_test, tahmin_yon)

print("\n=== SINIFLANDIRMA (Yön Tahmini) ===")
print(f"Accuracy (Doğruluk): {acc:.2%}")
print(f"F1 Score: {f1:.2%}")

=== REGRESYON (Fiyat Tahmini) ===
MAE  (Ortalama Mutlak Hata): 2064.30 TL
RMSE (Kök Ortalama Kare Hata): 2607.38 TL
Karşılaştırma - ortalama gram altın fiyatı: 4409.69 TL

=== SINIFLANDIRMA (Yön Tahmini) ===
Accuracy (Doğruluk): 49.58%
F1 Score: 48.37%


In [24]:
# Ham fiyat yerine yüzdesel değişimi hedefliyoruz
df["target_pct"] = ((df["target_fiyat"] - df["gram_altin_try"]) / df["gram_altin_try"]) * 100

df[["gram_altin_try", "target_fiyat", "target_pct"]].tail(10)

,gram_altin_try,target_fiyat,target_pct
Date,,,
2026-07-14,6139.613819,6114.177836,-0.414293
2026-07-15,6114.177836,6027.266076,-1.421479
2026-07-16,6027.266076,6076.698631,0.820149
2026-07-17,6076.698631,6078.097834,0.023026
2026-07-20,6078.097834,6175.412480,1.601071
2026-07-21,6175.412480,6292.672516,1.898821
2026-07-22,6292.672516,6142.385793,-2.388281
2026-07-23,6142.385793,6187.915087,0.741231
2026-07-24,6187.915087,6189.641682,0.027903


In [25]:
# NaN satırları yeniden temizle (target_pct de dahil)
df_model = df.dropna()

X = df_model[feature_kolonlari]
y_pct = df_model["target_pct"]

# Aynı tarih bazlı bölme
bolme_noktasi = int(len(X) * 0.8)
X_train, X_test = X.iloc[:bolme_noktasi], X.iloc[bolme_noktasi:]
y_pct_train, y_pct_test = y_pct.iloc[:bolme_noktasi], y_pct.iloc[bolme_noktasi:]

# Modeli yeniden eğit
model_pct = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
model_pct.fit(X_train, y_pct_train)

# Değerlendirme
tahmin_pct = model_pct.predict(X_test)
mae_pct = mean_absolute_error(y_pct_test, tahmin_pct)

print(f"MAE (yüzde cinsinden): {mae_pct:.3f}%")
print(f"Gerçek günlük değişimlerin ortalama mutlak büyüklüğü: {y_pct_test.abs().mean():.3f}%")


MAE (yüzde cinsinden): 1.190%
Gerçek günlük değişimlerin ortalama mutlak büyüklüğü: 1.037%


In [26]:
# Baseline 1: Her zaman "değişim olmayacak" (0%) diye tahmin et
baseline_sifir = np.zeros(len(y_pct_test))
mae_baseline_sifir = mean_absolute_error(y_pct_test, baseline_sifir)

# Baseline 2: "Yarın da bugünkü gibi değişecek" (dünkü değişim devam edecek varsayımı)
baseline_devam = y_pct_test.shift(1).fillna(0)
mae_baseline_devam = mean_absolute_error(y_pct_test, baseline_devam)

print(f"Model MAE:              {mae_pct:.3f}%")
print(f"Baseline (hep 0%):       {mae_baseline_sifir:.3f}%")
print(f"Baseline (dünkü devam):  {mae_baseline_devam:.3f}%")

Model MAE:              1.190%
Baseline (hep 0%):       1.037%
Baseline (dünkü devam):  1.562%


In [27]:
import pandas as pd

onem = pd.DataFrame({
    "feature": feature_kolonlari,
    "onem": model_pct.feature_importances_
}).sort_values("onem", ascending=False)

print(onem)

           feature      onem
9     volatilite_7  0.241507
0         gold_usd  0.125772
8            ma_30  0.103652
6  gram_altin_lag7  0.095225
1        usd_index  0.093147
3          oil_usd  0.088809
2            sp500  0.081074
4          usd_try  0.061372
5  gram_altin_lag1  0.056349
7             ma_7  0.053093


In [28]:
!pip install xgboost


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
model_xgb.fit(X_train, y_pct_train)

tahmin_xgb = model_xgb.predict(X_test)
mae_xgb = mean_absolute_error(y_pct_test, tahmin_xgb)

print(f"XGBoost MAE:             {mae_xgb:.3f}%")
print(f"Random Forest MAE:       {mae_pct:.3f}%")
print(f"Baseline (hep 0%):       {mae_baseline_sifir:.3f}%")
print(f"Baseline (dünkü devam):  {mae_baseline_devam:.3f}%")

XGBoost MAE:             1.141%
Random Forest MAE:       1.190%
Baseline (hep 0%):       1.037%
Baseline (dünkü devam):  1.562%


In [30]:
import joblib

joblib.dump(model_pct, "model_rf.pkl")
joblib.dump(model_xgb, "model_xgb.pkl")

# Test verisini de kaydedelim (grafiklerde kullanacağız)
X_test.to_csv("X_test.csv")
y_pct_test.to_csv("y_pct_test.csv")

# Baseline değerlerini de bir dosyaya yazalım
import json
sonuclar = {
    "mae_rf": float(mae_pct),
    "mae_xgb": float(mae_xgb),
    "mae_baseline_sifir": float(mae_baseline_sifir),
    "mae_baseline_devam": float(mae_baseline_devam),
    "feature_kolonlari": feature_kolonlari
}
with open("sonuclar.json", "w") as f:
    json.dump(sonuclar, f)

print("Kaydedildi: model_rf.pkl, model_xgb.pkl, X_test.csv, y_pct_test.csv, sonuclar.json")

Kaydedildi: model_rf.pkl, model_xgb.pkl, X_test.csv, y_pct_test.csv, sonuclar.json


In [31]:
"""
Altın Fiyatı Tahmin Projesi - Streamlit Arayüzü

Çalıştırmak için:
    streamlit run app.py

NOT: Bu dosyayı, notebook'ta kaydettiğin şu dosyalarla AYNI klasöre koy:
    model_rf.pkl, model_xgb.pkl, X_test.csv, y_pct_test.csv, sonuclar.json
"""

import streamlit as st
import pandas as pd
import joblib
import json

st.set_page_config(page_title="Altın Fiyatı Tahmin Projesi", layout="wide")

# ---- Verileri yükle ----
@st.cache_resource
def yukle():
    model_rf = joblib.load("model_rf.pkl")
    model_xgb = joblib.load("model_xgb.pkl")
    X_test = pd.read_csv("X_test.csv", index_col=0, parse_dates=True)
    y_test = pd.read_csv("y_pct_test.csv", index_col=0, parse_dates=True).iloc[:, 0]
    with open("sonuclar.json") as f:
        sonuclar = json.load(f)
    return model_rf, model_xgb, X_test, y_test, sonuclar

model_rf, model_xgb, X_test, y_test, sonuclar = yukle()

st.title("🪙 Altın Fiyatı Tahmin Projesi")
st.caption("Random Forest ve XGBoost ile günlük gram altın (TL) fiyat değişimi tahmini")

# ---- Üst kısım: model karşılaştırma metrikleri ----
st.subheader("Model karşılaştırması (MAE - Ortalama Mutlak Hata)")

col1, col2, col3, col4 = st.columns(4)
col1.metric("Random Forest", f"{sonuclar['mae_rf']:.3f}%")
col2.metric("XGBoost", f"{sonuclar['mae_xgb']:.3f}%")
col3.metric("Baseline (0%)", f"{sonuclar['mae_baseline_sifir']:.3f}%",
            delta="En iyi", delta_color="off")
col4.metric("Baseline (dünkü devam)", f"{sonuclar['mae_baseline_devam']:.3f}%")

st.info(
    "📌 **Dürüst bulgu:** Hiçbir model, en basit baseline'ı (her zaman "
    "'değişim olmayacak' varsayımı) geçemedi. Bu, günlük altın fiyat "
    "değişiminin kısa vadede büyük ölçüde öngörülemez olduğunu gösteriyor "
    "(etkin piyasa hipotezi ile tutarlı)."
)

st.divider()

# ---- Sekmeler: her model için ayrı detay ----
tab1, tab2, tab3 = st.tabs(["📊 Random Forest", "📊 XGBoost", "🔍 Feature Importance"])

with tab1:
    st.subheader("Random Forest — Tahmin vs Gerçek")
    tahmin_rf = model_rf.predict(X_test)
    karsilastirma_rf = pd.DataFrame({
        "Gerçek % değişim": y_test.values,
        "Tahmin % değişim": tahmin_rf
    }, index=X_test.index)
    st.line_chart(karsilastirma_rf)
    st.dataframe(karsilastirma_rf.tail(15), use_container_width=True)

with tab2:
    st.subheader("XGBoost — Tahmin vs Gerçek")
    tahmin_xgb = model_xgb.predict(X_test)
    karsilastirma_xgb = pd.DataFrame({
        "Gerçek % değişim": y_test.values,
        "Tahmin % değişim": tahmin_xgb
    }, index=X_test.index)
    st.line_chart(karsilastirma_xgb)
    st.dataframe(karsilastirma_xgb.tail(15), use_container_width=True)

with tab3:
    st.subheader("Hangi değişken modele ne kadar etki ediyor?")
    onem_df = pd.DataFrame({
        "feature": sonuclar["feature_kolonlari"],
        "Random Forest": model_rf.feature_importances_,
        "XGBoost": model_xgb.feature_importances_
    }).set_index("feature").sort_values("Random Forest", ascending=False)
    st.bar_chart(onem_df)

st.divider()

# ---- Canlı tahmin bölümü ----
st.subheader("🔮 Bugünün verisiyle canlı tahmin")
st.caption("En son satırdaki (bugünün) feature değerleriyle iki modelin tahminini karşılaştır")

son_satir = X_test.iloc[[-1]]
tahmin_rf_bugun = model_rf.predict(son_satir)[0]
tahmin_xgb_bugun = model_xgb.predict(son_satir)[0]

col1, col2 = st.columns(2)
col1.metric("Random Forest tahmini (yarınki % değişim)", f"{tahmin_rf_bugun:+.3f}%")
col2.metric("XGBoost tahmini (yarınki % değişim)", f"{tahmin_xgb_bugun:+.3f}%")

with st.expander("Kullanılan feature değerlerini gör"):
    st.dataframe(son_satir.T, use_container_width=True)

2026-07-25 15:52:21.283 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 15:52:21.284 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 15:52:22.392 
  command:

    streamlit run C:\Users\ibrah\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-07-25 15:52:22.393 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 15:52:22.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 15:52:22.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-25 15:52:22.896 Thread 'Thread-9': missing ScriptRunContext! This warning can be 

In [32]:
# X_test ile aynı tarihlere denk gelen gerçek gram altın fiyatlarını kaydediyoruz
gercek_fiyatlar = df_model.loc[X_test.index, "gram_altin_try"]
gercek_fiyatlar.to_csv("gercek_fiyatlar.csv")

print("Kaydedildi: gercek_fiyatlar.csv")
gercek_fiyatlar.tail()

Kaydedildi: gercek_fiyatlar.csv


Date
2026-07-20    6078.097834
2026-07-21    6175.412480
2026-07-22    6292.672516
2026-07-23    6142.385793
2026-07-24    6187.915087
Name: gram_altin_try, dtype: float64